In [1]:
%pip install -q fasttext-wheel langdetect pandas

Note: you may need to restart the kernel to use updated packages.


## Language detection from lyrics (FastText)

Reads `lyrics.csv`, detects the language of each song using the **first 200 characters** of its lyrics
via FastText's pre-trained language identification model (`lid.176.ftz`),
and writes `lyrics_lang.csv` to the same directory.

Output columns: `artist`, `title`, `language`, `spotify_uri`

In [2]:
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import fasttext
import fasttext.FastText

# Idempotent monkey-patch: fix np.array(copy=False) error in NumPy 2.x
if not hasattr(fasttext.FastText._FastText, '_orig_predict'):
    fasttext.FastText._FastText._orig_predict = fasttext.FastText._FastText.predict

    def _safe_predict(self, text, k=1, threshold=0.0, on_unicode_error='strict'):
        import numpy as np
        _old_array = np.array
        def _array_compat(*args, **kwargs):
            kwargs.pop('copy', None)
            return _old_array(*args, **kwargs)
        np.array = _array_compat
        try:
            return fasttext.FastText._FastText._orig_predict(self, text, k=k, threshold=threshold, on_unicode_error=on_unicode_error)
        finally:
            np.array = _old_array

    fasttext.FastText._FastText.predict = _safe_predict
    print('NumPy 2.x patch applied.')
else:
    print('Patch already applied, skipping.')

# ── Download FastText lid model if not cached ────────────────────────────────
MODEL_URL = 'https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz'
MODEL_PATH = Path('..') / 'models' / 'lid.176.ftz'
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    print(f'Downloading FastText lid model \u2192 {MODEL_PATH} ...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print('Done.')
else:
    print(f'Model already cached at {MODEL_PATH}')

ft_model = fasttext.load_model(str(MODEL_PATH))
print('FastText model loaded.')

NumPy 2.x patch applied.
Model already cached at ../models/lid.176.ftz
FastText model loaded.


In [3]:
DATE_PATH = '2026/03/05'
LYRICS_DIR = Path('..') / 'data' / 'processed' / 'lyrics' / DATE_PATH
LYRICS_IN  = LYRICS_DIR / 'lyrics.csv'
LANG_OUT   = LYRICS_DIR / 'lyrics_lang.csv'

df = pd.read_csv(LYRICS_IN)
print(f'Loaded {len(df)} songs from {LYRICS_IN}')

def detect_language_ft(lyrics: str, n_chars: int = 200) -> str:
    """
    Detect language from the first `n_chars` characters of lyrics
    using FastText lid.176.ftz. Returns ISO 639-1 code or 'unknown'.
    """
    if not isinstance(lyrics, str) or not lyrics.strip():
        return 'unknown'
    # FastText expects single-line input
    snippet = lyrics[:n_chars].replace('\n', ' ').strip()
        return 'unknown'
    try:
        result = ft_model.predict(snippet)
        label = result[0][0]
        return label.replace('__label__', '')
    except Exception:
        return 'unknown'

df['language'] = df['lyrics'].apply(detect_language_ft)

out_cols = ['artist', 'title', 'region', 'language']
if 'spotify_uri' in df.columns:
    out_cols.append('spotify_uri')

lang_df = df[out_cols].copy()
lang_df.to_csv(LANG_OUT, index=False)

print(f'Saved {len(lang_df)} rows \u2192 {LANG_OUT}')
print('\nLanguage distribution:')
print(df['language'].value_counts().to_string())
lang_df.head(10)

Loaded 400 songs from ../data/processed/lyrics/2026/03/05/lyrics.csv
Saved 400 rows → ../data/processed/lyrics/2026/03/05/lyrics_lang.csv

Language distribution:
language
en         199
es         109
unknown     35
zh          27
ko          12
tr           5
pt           2
it           2
ru           2
ja           2
he           1
id           1
vi           1
gd           1
de           1


,artist,title,region,language,spotify_uri
0,"Mr Plata, El Americano 4KT",Las Muñequitas,Colombia,es,4nJJCRYru4QQakCiUA155f
1,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),Colombia,es,3CBEVPwR3kUXDoTx1lqFUQ
2,"Ryan Castro, Kapo, Gangsta",LA VILLA,Colombia,es,2ZyrAym0sRLwt4PhGotHuI
3,Kris R.,GANAS,Colombia,es,4KE9Ne3hgh18B3Th4xcylg
4,"W Sound, Beéle, Ovy On The Drums",La Plena - W Sound 05,Colombia,en,6iOndD4OFo7GkaDypWQIou
5,Beéle,no tiene sentido,Colombia,es,1HEwEN64NjgTaHmo7LfkX8
6,"J Balvin, Ryan Castro, DJ Snake",Tonto,Colombia,es,7mU1fei7P9h4mpjP2Otdw5
7,Beéle,quédate,Colombia,es,6VfL3MEuYeJbDlD8m011HR
8,Grupo Firme,El Beneficio De La Duda,Colombia,es,5yXt80BNZGbmHFd0NHZHNn
9,"Yeison Jimenez, Luis Alfonso",Destino Final,Colombia,es,2E4TYekUduml1DWIqQWNcj


In [ ]:
# LANG_FT_OUT = LYRICS_DIR / 'lyrics_lang.csv'

# out_cols = ['artist', 'title', 'language']
# if 'spotify_uri' in df.columns:
#     out_cols.append('spotify_uri')

# lang_ft_df = df[out_cols].copy()
# lang_ft_df.to_csv(LANG_FT_OUT, index=False)

# print(f'FastText: saved {len(lang_ft_df)} rows \u2192 {LANG_FT_OUT}')
# lang_ft_df.head(10)

FastText: saved 400 rows → ../data/processed/lyrics/2026/03/05/lyrics_lang_fasttext.csv


,artist,title,language,spotify_uri
0,"Mr Plata, El Americano 4KT",Las Muñequitas,unknown,4nJJCRYru4QQakCiUA155f
1,"ARIA VEGA, Ryan Castro",CHÉVERE (premium_remix),unknown,3CBEVPwR3kUXDoTx1lqFUQ
2,"Ryan Castro, Kapo, Gangsta",LA VILLA,unknown,2ZyrAym0sRLwt4PhGotHuI
3,Kris R.,GANAS,unknown,4KE9Ne3hgh18B3Th4xcylg
4,"W Sound, Beéle, Ovy On The Drums",La Plena - W Sound 05,unknown,6iOndD4OFo7GkaDypWQIou
5,Beéle,no tiene sentido,unknown,1HEwEN64NjgTaHmo7LfkX8
6,"J Balvin, Ryan Castro, DJ Snake",Tonto,unknown,7mU1fei7P9h4mpjP2Otdw5
7,Beéle,quédate,unknown,6VfL3MEuYeJbDlD8m011HR
8,Grupo Firme,El Beneficio De La Duda,unknown,5yXt80BNZGbmHFd0NHZHNn
9,"Yeison Jimenez, Luis Alfonso",Destino Final,unknown,2E4TYekUduml1DWIqQWNcj


---
## langdetect comparison

Runs the same detection using `langdetect` (Google's language-detection port) on the first 200 characters.
Writes `lyrics_lang_langdetect.csv` and compares both methods.

In [19]:
# from langdetect import detect, LangDetectException

# def detect_language_ld(lyrics: str, n_chars: int = 200) -> str:
#     """
#     Detect language from the first `n_chars` characters of lyrics
#     using langdetect. Returns ISO 639-1 code or 'unknown'.
#     """
#     if not isinstance(lyrics, str) or not lyrics.strip():
#         return 'unknown'
#     snippet = lyrics[:n_chars].replace('\n', ' ').strip()
#     if not snippet:
#         return 'unknown'
#     try:
#         return detect(snippet)
#     except LangDetectException:
#         return 'unknown'

# df['language_ld'] = df['lyrics'].apply(detect_language_ld)

# LANG_LD_OUT = LYRICS_DIR / 'lyrics_lang_langdetect.csv'
# out_cols_ld = ['artist', 'title', 'language_ld']
# if 'spotify_uri' in df.columns:
#     out_cols_ld.append('spotify_uri')

# lang_ld_df = df[out_cols_ld].rename(columns={'language_ld': 'language'}).copy()
# lang_ld_df.to_csv(LANG_LD_OUT, index=False)

# print(f'langdetect: saved {len(lang_ld_df)} rows \u2192 {LANG_LD_OUT}')
# print('\nlangdetect distribution:')
# print(lang_ld_df['language'].value_counts().to_string())

---
## Side-by-side comparison

In [20]:
# comparison = df[['artist', 'title', 'language', 'language_ld']].rename(
#     columns={'language': 'fasttext', 'language_ld': 'langdetect'}
# )
# comparison['agree'] = comparison['fasttext'] == comparison['langdetect']

# agree_pct = comparison['agree'].mean() * 100
# print(f'Agreement: {agree_pct:.1f}% ({comparison["agree"].sum()}/{len(comparison)})')

# print(f'\nDisagreements ({(~comparison["agree"]).sum()} songs):')
# print(comparison[~comparison['agree']].to_string())

# print('\n--- FastText distribution ---')
# print(comparison['fasttext'].value_counts().to_string())
# print('\n--- langdetect distribution ---')
# print(comparison['langdetect'].value_counts().to_string())